# Imports

In [ ]:
import ast
import os
import pandas as pd
from sqlalchemy import create_engine, text
from tqdm import tqdm
import pyodbc
from dotenv import load_dotenv

In [ ]:
# Check available driver versions
print("Available ODBC Drivers:")
for driver in pyodbc.drivers():
  print(f" - {driver}")

# 1. Database Connection (Windows Authentication)

In [ ]:
load_dotenv()
server_name = os.getenv("SERVER_NAME")
database_name = os.getenv("DATABASE_NAME")
# Change driver version below if required
conn_str = f"mssql+pyodbc://{server_name}/{database_name}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes&encrypt=no"
# Otherwise use
# conn_str = os.getenv("CONN_STR")

engine = create_engine(conn_str)

# 2. Load Master Metadata CSV

In [ ]:
csv_file = "yahoo_metadata_all.csv"  # Ensure your rebuilt CSV is here
print(f"Loading metadata from {csv_file}...")
df_meta = pd.read_csv(csv_file)
print(f"Loaded {len(df_meta):,} rows from metadata CSV.")

# 3. Populate Folders Table

In [ ]:
print("Extracting and inserting unique folders...")
unique_folders = df_meta["folder"].dropna().unique()
folder_df = pd.DataFrame({"FolderName": unique_folders})

# Insert folders and fetch their generated IDs back
with engine.begin() as conn:
  # Clear existing data if re-running
  conn.execute(text("DELETE FROM Attachments"))
  conn.execute(text("DELETE FROM Emails"))
  conn.execute(text("DELETE FROM Folders"))

  # Wrapped folder insertion in a progress tracker
  for i in tqdm(range(0, len(folder_df), 1000), desc="Loading Folders"):
    chunk = folder_df.iloc[i : i + 1000]
    chunk.to_sql(
        "Folders",
        con=conn,
        if_exists="append",
        index=False,
        method="multi",
    )

# Fetch FolderID mappings from DB
folder_mapping = pd.read_sql(
    "SELECT FolderID, FolderName FROM Folders", engine
)
folder_dict = dict(
    zip(folder_mapping["FolderName"], folder_mapping["FolderID"])
)

# Map FolderID into our main dataframe
df_meta["FolderID"] = df_meta["folder"].map(folder_dict)

# 4. Populate Emails table

In [ ]:
chunk_size = 5000
total_chunks = (len(df_meta) + chunk_size - 1) // chunk_size

print(
    f"Streaming Emails into SQL Server in {total_chunks} chunks (5,000 rows"
    " each)..."
)

with engine.begin() as conn:
  for i in tqdm(
      range(0, len(df_meta), chunk_size),
      desc="Loading Email Chunks",
      unit="chunk",
  ):
    chunk = df_meta.iloc[i : i + chunk_size].copy()

    emails_chunk = pd.DataFrame({
        "FolderID": chunk["FolderID"],
        "EmailUID": chunk["uid"].astype(str),
        "Sender": chunk["sender"],
        "Recipient": chunk["recipient"],
        "Cc": chunk["cc"],
        "Bcc": chunk["bcc"],
        "EmailDate": chunk["date"],
        "EmailSubject": chunk["subject"],
        "HasAttachments": chunk["has_attachments"].astype(int),
        "AttachmentExtensions": chunk["attachment_extensions"],
        "LocalEmlPath": chunk["eml_path"],
        "BodySnippet": chunk["body_snippet"],
    })

    emails_chunk.to_sql(
        "Emails",
        con=conn,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=100,
    )

print("✅ Emails loaded successfully!")

# 5. Populate Attachments table

In [ ]:
print("Extracting and loading child attachments...")

# Fetch EmailID and EmailUID mappings to tie attachments back correctly
email_mapping = pd.read_sql(
    "SELECT EmailID, FolderID, EmailUID FROM Emails", engine
)
# Create a lookup map: (FolderID, EmailUID) -> EmailID
email_dict = {}
for row in email_mapping.itertuples():
  email_dict[(row.FolderID, row.EmailUID)] = row.EmailID

attachment_records = []
for row in tqdm(
    df_meta.itertuples(), total=len(df_meta), desc="Parsing Attachments"
):
  if row.has_attachments == 1 and pd.notna(row.attachment_names):
    try:
      names_list = ast.literal_eval(row.attachment_names)
      f_id = row.FolderID
      e_uid = str(row.uid)
      db_email_id = email_dict.get((f_id, e_uid))

      if db_email_id:
        for filename in names_list:
          _, ext = os.path.splitext(filename)
          # Reconstruct local attachment path if applicable
          safe_folder_name = "".join(
              c if c.isalnum() else "_" for c in str(row.folder)
          )
          attach_dir = os.path.join(
              ATTACH_DIR if "ATTACH_DIR" in globals() else "./attachments",
              f"email_{safe_folder_name}_{e_uid}",
          )
          local_path = os.path.join(attach_dir, filename)

          attachment_records.append({
              "EmailID": db_email_id,
              "OriginalFileName": filename,
              "FileExtension": ext.lower() if ext else None,
              "LocalAttachmentPath": local_path,
          })
    except Exception:
      continue

# Bulk insert attachments in chunks if any exist
if attachment_records:
  df_attachments = pd.DataFrame(attachment_records)
  with engine.begin() as conn:
    for i in tqdm(
        range(0, len(df_attachments), chunk_size),
        desc="Loading Attachment Chunks",
        unit="chunk",
    ):
      chunk = df_attachments.iloc[i : i + chunk_size]
      chunk.to_sql(
          "Attachments",
          con=conn,
          if_exists="append",
          index=False,
          method="multi",
          chunksize=100,
      )
  print(f"✅ Loaded {len(df_attachments):,} attachment records!")
else:
  print("ℹ️ No attachments found to load.")

print("🎉 Complete bulk load finished successfully!")